In [ ]:
import pandas as pd
import numpy as np
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import euclidean_distances
from sklearn.metrics.pairwise import linear_kernel
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics.pairwise import manhattan_distances

In [ ]:
data = pd.read_csv(r'movies.csv', delimiter=';')
data.dropna(subset=['actors','director','writer','original_title','description','genre'],inplace=True,axis=0)
data = data.reset_index(drop=True)

data["combined"] = data['genre'] + '  ' + data['actors'] + ' ' + data['director'] + ' ' + data['writer'] + ' ' + data['original_title'] + ' ' + data['description']

# Because there is a new row created with the previous columns they now are going to be dropped
data.drop(['actors','director','writer','description','genre'],axis=1,inplace=True)

In [ ]:
# Gets every word from the "combined" column and adds it to a list
vectorizer = TfidfVectorizer(analyzer='word', stop_words='english')
matrix = vectorizer.fit_transform(data["combined"])

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
count = CountVectorizer(stop_words='english')
count_matrix = count.fit_transform(data['soup'])

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

cosine_sim2 = cosine_similarity(count_matrix, count_matrix)
movie_title = data['original_title']
indices = pd.Series(data.index, index=data['original_title'])

In [ ]:
%%time
linear_kernel = linear_kernel(matrix,matrix)
cosine_similarities = cosine_similarity(matrix,matrix)
euclidean_similarities = euclidean_distances(matrix,matrix)
manhattan_similarities = manhattan_distances(matrix, matrix)

movie_title = data['original_title']
indices = pd.Series(data.index, index=data['original_title'])

CPU times: user 1min 56s, sys: 1min 2s, total: 2min 58s
Wall time: 2min 58s


In [ ]:
# Function to sort the movies based on similarity
def content_recommender(title):
    idx = indices[title]
    # sim_scores = list(enumerate(linear_kernel[idx]))
    # sim_scores = list(enumerate(euclidean_similarities[idx]))
    # sim_scores = list(enumerate(cosine_similarities[idx]))
    # sim_scores = list(enumerate(manhattan_similarities[idx]))
    sim_scores = list(enumerate(cosine_sim2[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:11]
    movie_indices = [i[0] for i in sim_scores]
    return movie_title.iloc[movie_indices]

In [ ]:
print(content_recommender('The Dark Knight'))

NameError: name 'content_recommender' is not defined

# Sentence Transformers

In [ ]:
!pip install sentence_transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 345.7/345.7 kB 6.4 MB/s eta 0:00:00


In [ ]:
data = pd.read_csv(r'movies.csv', delimiter=';')
data.dropna(subset=['original_title','description','genre'],inplace=True,axis=0)
data = data.reset_index(drop=True)

data["combined"] = data['genre'] + ' ' + data['original_title'] + ' ' + data['description']

# Because there is a new row created with the previous columns they now are going to be dropped
data.drop(['description','genre'],axis=1,inplace=True)

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-mpnet-base-v2')
movie_descriptions = data['combined'].tolist()

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.4k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
%%time
movie_descriptions = data['combined'].tolist()
embeddings = model.encode(movie_descriptions)

CPU times: user 7h 25min 53s, sys: 3min 31s, total: 7h 29min 24s
Wall time: 9min 15s


In [ ]:
# linear_kernel = linear_kernel(embeddings)
# cosine_similarities = cosine_similarity(embeddings, embeddings)
cosine_similarities = cosine_similarity(embeddings)

In [ ]:
print(cosine_similarities.shape)

(83740, 83740)


In [ ]:
movie_title = data['original_title']
indices = pd.Series(data.index, index=data['original_title'])

def content_recommender(title):
    idx = indices[title]
    # sim_scores = list(enumerate(linear_kernel[idx]))
    sim_scores = list(enumerate(cosine_similarities[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:11]
    movie_indices = [i[0] for i in sim_scores]
    return movie_title.iloc[movie_indices]

In [ ]:
print(content_recommender('The Godfather'))

58057         The Last Godfather
30895       Jane Austen's Mafia!
16401     The Godfather: Part II
13300            The Brotherhood
67363              Príbeh kmotra
51851                      Mafia
25022    The Godfather: Part III
15776                    Il boss
27885            Lookin' Italian
38871            Avenging Angelo
Name: original_title, dtype: object


In [ ]:
print(content_recommender('Avengers: Endgame'))

71919             Avengers: Infinity War
49463                       The Avengers
65183            Avengers: Age of Ultron
30009                       The Avengers
77837    Avengers of Justice: Farce Wars
69558         Captain America: Civil War
55164                     Iron Man Three
54211                         Iron Man 2
69580                     Thor: Ragnarok
43139              X-Men: The Last Stand
Name: original_title, dtype: object


In [ ]:
print(content_recommender('Spider-Man'))

50370               The Amazing Spider-Man
44566                         Spider-Man 3
61122             The Amazing Spider-Man 2
78290            Spider-Man: Far from Home
63960               Spider-Man: Homecoming
40892                         Spider-Man 2
4001                    The Spider Returns
66858               Spider Man: Lost Cause
73291    Spider-Man: Into the Spider-Verse
60696                      Big Ass Spider!
Name: original_title, dtype: object


In [ ]:
print(content_recommender('James Bond'))

58198                          Nems Bond
65010                            Spectre
52087                            Skyfall
14011    On Her Majesty's Secret Service
24455                    Licence to Kill
24741                     Vanille fraise
40828                        The In-Laws
22640                     Something Wild
12856                      Casino Royale
80379                   Spy Intervention
Name: original_title, dtype: object
